# Live exercise: from notebook code to a pipeline, worked solution

The finished script is `sessions/session-10/solutions/device_pipeline.py`.
This notebook builds the same functions one at a time, with the checks.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

MESSY = Path("data/messy/plays_messy.csv")
OUTPUT = Path("output")

## 1. `load_messy(path)`

In [ ]:
def load_messy(path):
    """Return the messy CSV at path as a DataFrame, with nothing guessed.

    keep_default_na=False stops pandas turning "NA" and empty cells into
    missing values on its own. clean_plays decides what counts as missing.
    """
    return pd.read_csv(path, keep_default_na=False)


raw = load_messy(MESSY)
assert raw.shape == (2223, 9)

One line, and still worth being a function. When your own data arrives as
an Excel file or needs `sep=";"`, this is the only place that changes.

## 2. `clean_plays(df)`

The session 9 function, unchanged, with its helpers: the same cell as
notebook 01.

In [ ]:
# The column order of data/clean/plays.csv.
COLUMNS = [
    "play_id", "played_at", "artist_name", "track_name", "genre",
    "country", "minutes_played", "device", "skipped",
]

# The three ways a date is written in the messy file, tried in this order.
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"]


def parse_dates(text):
    """Return a column of dates written in any of DATE_FORMATS as datetimes.

    Each format is tried on purpose, one at a time. A single blanket parse
    reads 05/01/2024 as the 1st of May, without a word of warning.
    """
    dates = pd.to_datetime(text, format=DATE_FORMATS[0], errors="coerce")
    for date_format in DATE_FORMATS[1:]:
        dates = dates.fillna(pd.to_datetime(text, format=date_format, errors="coerce"))
    assert dates.notna().all(), "some dates are in none of DATE_FORMATS"
    return dates


def fill_from_artist(df, column):
    """Return column with its gaps filled from the same artist's other rows.

    Every artist has exactly one genre and one country, so a play with the
    country missing can borrow it from any other play by the same artist.
    This is the session 4 lookup dictionary, built with groupby.
    """
    lookup = df.dropna(subset=[column]).groupby("artist_name")[column].first()
    return df[column].fillna(df["artist_name"].map(lookup))


def clean_plays(df):
    """Return a cleaned copy of the messy plays, shaped like data/clean/plays.csv.

    The decisions, in order:
      * the "minutes played " column is renamed to minutes_played
      * exact duplicate rows are dropped (40 in the messy file), first,
        because they are exact copies and every later step is then smaller
      * genre is stripped and title-cased, device stripped and lower-cased
      * dates in three formats are parsed deliberately, then written back as
        YYYY-MM-DD text, which is how the clean file stores them
      * " min" and spaces are removed from minutes before converting; a blank
        stays missing (NaN), because nobody knows how long that play was
      * "NA" in country means missing here (no artist is from Namibia), and
        missing genre and country are filled from the artist's other plays
      * a missing device becomes "unknown", so those plays still get counted

    The input DataFrame is not changed.
    """
    df = df.rename(columns={"minutes played ": "minutes_played"})
    df = df.drop_duplicates().copy()
    assert df["play_id"].is_unique, "a play_id appears twice with different values"

    # Work from text in every damaged column, so this behaves the same even if
    # the file was read without keep_default_na=False.
    text_columns = ["played_at", "genre", "country", "device", "minutes_played"]
    df[text_columns] = df[text_columns].fillna("").astype(str)

    df["genre"] = df["genre"].str.strip().str.title().replace("", pd.NA)
    df["device"] = df["device"].str.strip().str.lower().replace("", "unknown")
    df["country"] = df["country"].str.strip().replace(["NA", ""], pd.NA)
    df["genre"] = fill_from_artist(df, "genre")
    df["country"] = fill_from_artist(df, "country")

    df["played_at"] = parse_dates(df["played_at"]).dt.strftime("%Y-%m-%d")

    # Remove " min", and any spaces: the export writes thousands as "1 234.50".
    # No play in this file is that long, but nothing else would catch it.
    minutes = df["minutes_played"].str.replace("min", "").str.replace(" ", "")
    df["minutes_played"] = pd.to_numeric(minutes, errors="coerce")

    # errors="coerce" never complains, so check it only blanked the blanks.
    unreadable = df["minutes_played"].isna() & (minutes != "")
    assert not unreadable.any(), "some minutes could not be read as numbers"

    return df[COLUMNS].sort_values("play_id").reset_index(drop=True)


plays = clean_plays(raw)
assert len(plays) == 2183
assert (plays["device"] == "unknown").sum() == 69

## 3. `skip_rate_by_device(plays)`

In [ ]:
def skip_rate_by_device(plays):
    """Return one row per device: plays, skipped, and the skip rate in percent."""
    rates = (plays.groupby("device")
             .agg(plays=("play_id", "count"), skipped=("skipped", "sum"))
             .reset_index())
    rates["skip_rate"] = (rates["skipped"] / rates["plays"] * 100).round(1)
    return rates.sort_values("skip_rate", ascending=False).reset_index(drop=True)


rates = skip_rate_by_device(plays)
assert len(rates) == 6
assert rates["plays"].sum() == 2183
assert rates["skipped"].sum() == 258
rates

### The top row is not a device

The highest skip rate belongs to `unknown`: 11 skips in 69 plays, 15.9%.
Those are the plays whose device was blank in the messy file. We labelled
them in `clean_plays`. So the "winner" is a category made by our own
cleaning decision, and a chart that ranked it first would be answering a
question nobody asked.

The fix is not to delete them from the data. The table should still add up
to every play, which is exactly what the `assert` above checks. Instead,
the **chart** leaves `unknown` out, because the chart answers the question
about devices. The numbers stay complete; the picture stays on topic.

In [ ]:
assert rates.loc[0, "device"] == "unknown"
assert rates.loc[1, "device"] == "speaker"
assert rates.loc[1, "skip_rate"] == 14.1

## 4. `save_csv` and `save_chart`

In [ ]:
def save_csv(table, path):
    """Write a table to path as a CSV, making the folder if needed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False)


def save_chart(rates, path):
    """Draw the skip rate per real device as a bar chart, titled with the answer."""
    known = rates[rates["device"] != "unknown"]
    top = known.iloc[0]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(known["device"], known["skip_rate"], color="#4f6ddb")
    ax.invert_yaxis()
    ax.bar_label(bars, labels=[f"{r:.1f}%" for r in known["skip_rate"]], padding=3)
    ax.set_title(f"The {top['device']} gets skipped most: {top['skip_rate']:.1f}% of plays")
    ax.set_xlabel("plays skipped, %")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)


save_csv(rates, OUTPUT / "device_skip_rate.csv")
save_chart(rates, OUTPUT / "device_skip_rate.png")

In [ ]:
from IPython.display import Image

Image(filename="output/device_skip_rate.png", width=640)

`save_csv` takes any table, not only this one, which is why its parameter
is called `table` rather than `rates`. The homework reuses it unchanged.

## 5. `check_rates(path, plays)`

In [ ]:
def check_rates(path, plays):
    """Read the written CSV back and confirm it agrees with the cleaned plays."""
    back = pd.read_csv(path)
    assert back["plays"].sum() == len(plays), "plays went missing on the way"
    assert back["skipped"].sum() == plays["skipped"].sum(), "skips went missing"


check_rates(OUTPUT / "device_skip_rate.csv", plays)

Silence means it passed. To see it work, break it on purpose: hand it only
half of the plays, and the check catches it.

In [ ]:
try:
    check_rates(OUTPUT / "device_skip_rate.csv", plays.head(1000))
except AssertionError as error:
    print("caught:", error)

(`try` and `except` catch the error so the notebook keeps going. Session 12
covers them properly; here they only stop the demonstration from halting
the notebook.)

## 6. `main()`

In [ ]:
def main():
    """Run the pipeline. The outer layer: the only function that prints."""
    raw = load_messy(MESSY)
    plays = clean_plays(raw)
    rates = skip_rate_by_device(plays)

    csv_path = OUTPUT / "device_skip_rate.csv"
    png_path = OUTPUT / "device_skip_rate.png"
    save_csv(rates, csv_path)
    save_chart(rates, png_path)
    check_rates(csv_path, plays)

    print(f"{len(raw)} rows in, {len(plays)} after cleaning")
    print(rates.to_string(index=False))
    print(f"wrote {csv_path.as_posix()} and {png_path.as_posix()}")


main()

## 7. The script, from three folders

In the script, the paths come from `__file__` instead of being relative:

```python
# parents[0] is solutions, [1] session-10, [2] sessions, [3] the repo root.
ROOT = Path(__file__).resolve().parents[3]
MESSY = ROOT / "data" / "messy" / "plays_messy.csv"
OUTPUT = ROOT / "output"
```

Everything else is the functions above, plus the two lines at the bottom.

In [ ]:
import subprocess
import sys

script = Path("sessions/session-10/solutions/device_pipeline.py").resolve()
outputs = []
for folder in [".", "sessions/session-10", "data/clean"]:
    result = subprocess.run([sys.executable, str(script)], cwd=folder,
                            capture_output=True, text=True, check=True)
    outputs.append(result.stdout)
    print(f"from {folder:20} -> {result.stdout.splitlines()[0]}")

assert outputs[0] == outputs[1] == outputs[2]
print()
print(outputs[0])

Three folders, identical output, checked by an `assert` rather than by eye.

## If you have time: import it and reuse it

In [ ]:
sys.path.insert(0, "sessions/session-10/solutions")

import device_pipeline

in_2025 = plays[plays["played_at"].str.startswith("2025")]
device_pipeline.skip_rate_by_device(in_2025)

Nothing printed on import: `__name__` was `"device_pipeline"`, so `main()`
did not run. And the function took a different slice of the data without
a single change, because it was written to take `plays` as a parameter.

Look at what it says, though. In 2025 the tablet comes top at 17.3%, which
is 9 skips in 52 plays. In 2024 the tablet was bottom, at 10.2% (5 in 49).
A handful of skips moves a small device from last to first. Same lesson as
the session 4 stretch: a rate built on a few dozen plays deserves
suspicion, and the play count belongs next to the rate in every table.

In [ ]:
in_2024 = plays[plays["played_at"].str.startswith("2024")]
by_2024 = device_pipeline.skip_rate_by_device(in_2024)
assert by_2024.iloc[-1]["device"] == "tablet"
by_2024